# 08 Model Saving and Loading

This notebook saves the best PM2.5 model and the preprocessing bundle so it can be reused later for deployment or prediction.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

In [2]:
BASE_DIR = Path.cwd()
ARTIFACTS_DIR = BASE_DIR / "artifacts"
DEPLOYMENT_DIR = BASE_DIR / "deployment_assets"
DEPLOYMENT_DIR.mkdir(exist_ok=True)

In [3]:
results_df = pd.read_csv(ARTIFACTS_DIR / "pm25_model_training_results.csv")
results_df

,model,r2_score,mae,rmse
0,Random Forest,0.820760,22.815024,33.194107
1,Extra Trees,0.815294,23.430699,33.696378
2,AdaBoost,0.801368,25.112217,34.943624
3,XGBoost,0.799729,24.561260,35.087529
4,Gradient Boosting,0.798571,24.375944,35.188774
5,SVR,0.762164,26.967430,38.236814
6,KNN,0.760043,27.288043,38.406947


In [4]:
best_model_name = results_df.iloc[0]["model"]
safe_name = best_model_name.lower().replace(" ", "_")

best_model = joblib.load(ARTIFACTS_DIR / f"{safe_name}_model.joblib")
preprocessor = joblib.load(ARTIFACTS_DIR / "pm25_preprocessor.joblib")
feature_columns = preprocessor.feature_names_in_.tolist()

print("Best model selected for saving:", best_model_name)

Best model selected for saving: Random Forest


In [5]:
deployment_bundle = {
    "model_name": best_model_name,
    "model": best_model,
    "preprocessor": preprocessor,
    "feature_columns": feature_columns,
    "target_name": "Daily_PM25",
    "required_history_length": 30,
    "metrics": results_df.iloc[0].to_dict()
}

bundle_path = DEPLOYMENT_DIR / "best_pm25_model_bundle.joblib"
joblib.dump(deployment_bundle, bundle_path)

print("Saved bundle to:", bundle_path)

Saved bundle to: c:\Users\Anshul\Desktop\ml sem 4 mini project\deployment_assets\best_pm25_model_bundle.joblib


## Load the Saved Bundle

In [6]:
loaded_bundle = joblib.load(bundle_path)

print("Loaded model name:", loaded_bundle["model_name"])
print("Target name:", loaded_bundle["target_name"])
print("Required history length:", loaded_bundle["required_history_length"])
print("Best model metrics:")
print(loaded_bundle["metrics"])

Loaded model name: Random Forest
Target name: Daily_PM25
Required history length: 30
Best model metrics:
{'model': 'Random Forest', 'r2_score': 0.8207597898015471, 'mae': 22.815023688546717, 'rmse': 33.19410733041399}


## Quick Prediction Check After Loading

In [7]:
sample_input = pd.DataFrame([{
    "Avg_Temperature": 24.5,
    "Max_Temperature": 30.8,
    "Min_Temperature": 19.6,
    "Humidity": 58.0,
    "Rainfall_Snowmelt": 0.0,
    "Visibility": 6.8,
    "Wind_Speed": 4.2,
    "Max_Sustained_Wind_Speed": 7.4,
    "City": "Bangalore",
    "Month": 11,
    "Year": 2015,
    "Day": 20,
    "DayOfWeek": 4,
    "IsWeekend": 0,
    "Season": "Post-Monsoon",
    "PM25_lag_1": 48.9,
    "PM25_lag_2": 49.5,
    "PM25_lag_3": 50.6,
    "PM25_lag_5": 52.7,
    "PM25_lag_7": 54.1,
    "PM25_lag_10": 57.3,
    "PM25_lag_14": 61.9,
    "PM25_lag_21": 69.7,
    "PM25_lag_30": 72.5,
    "PM25_roll_mean_3": (48.9 + 49.5 + 50.6) / 3,
    "PM25_roll_mean_5": (48.9 + 49.5 + 50.6 + 51.8 + 52.7) / 5,
    "PM25_roll_mean_7": (48.9 + 49.5 + 50.6 + 51.8 + 52.7 + 53.9 + 54.1) / 7,
    "PM25_roll_mean_10": (48.9 + 49.5 + 50.6 + 51.8 + 52.7 + 53.9 + 54.1 + 55.8 + 57.3 + 58.6) / 10,
    "PM25_roll_mean_14": (48.9 + 49.5 + 50.6 + 51.8 + 52.7 + 53.9 + 54.1 + 55.8 + 57.3 + 58.6 + 59.8 + 60.1 + 61.9 + 62.2) / 14,
    "PM25_roll_mean_21": (48.9 + 49.5 + 50.6 + 51.8 + 52.7 + 53.9 + 54.1 + 55.8 + 57.3 + 58.6 + 59.8 + 60.1 + 61.9 + 62.2 + 63.7 + 64.8 + 66.1 + 67.5 + 69.7 + 71.8 + 74.2) / 21,
    "PM25_roll_mean_30": (48.9 + 49.5 + 50.6 + 51.8 + 52.7 + 53.9 + 54.1 + 55.8 + 57.3 + 58.6 + 59.8 + 60.1 + 61.9 + 62.2 + 63.7 + 64.8 + 66.1 + 67.5 + 69.7 + 71.8 + 74.2 + 76.3 + 80.0 + 82.1 + 79.4 + 74.6 + 68.0 + 70.2 + 75.1 + 72.5) / 30,
    "PM25_change_1": 48.9 - 49.5,
    "PM25_change_3": 48.9 - 51.8,
    "Temp_Range": 30.8 - 19.6,
    "Month_sin": np.sin(2 * np.pi * 11 / 12),
    "Month_cos": np.cos(2 * np.pi * 11 / 12)
}])[loaded_bundle["feature_columns"]]

sample_prediction = loaded_bundle["model"].predict(
    loaded_bundle["preprocessor"].transform(sample_input)
)[0]

print(f"Sample predicted PM2.5: {sample_prediction:.2f}")

Sample predicted PM2.5: 50.87
